# TP2: Spatial Filtering and Fourier Transform

Noise simulation, low/high-pass and median filters, sharpening, and frequency-domain analysis.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline


## 1. Load image as grayscale

In [ ]:
A = cv2.imread("image.jpg", cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(6, 6))
plt.imshow(A, cmap="gray")
plt.title("Original")
plt.axis("off")
plt.show()


## 2. Add noise

**Salt-and-pepper**: randomly forces 2% of pixels to 0 or 255.

**Gaussian**: zero-mean additive noise with sigma=10.

In [ ]:
rng = np.random.default_rng(42)

noisy_sp = A.copy().astype(np.float64)
noisy_sp[rng.random(A.shape) < 0.02] = 255
noisy_sp[rng.random(A.shape) < 0.02] = 0
noisy_sp = noisy_sp.astype(np.uint8)

noisy_gaussian = np.clip(A.astype(np.float64) + rng.normal(0, 10, A.shape), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(noisy_sp, cmap="gray"); axes[0].set_title("Salt-and-Pepper"); axes[0].axis("off")
axes[1].imshow(noisy_gaussian, cmap="gray"); axes[1].set_title("Gaussian"); axes[1].axis("off")
plt.show()


## 3. Low-pass (Gaussian blur) and median filtering

In [ ]:
low_sp  = cv2.GaussianBlur(noisy_sp, (5, 5), 0)
low_g   = cv2.GaussianBlur(noisy_gaussian, (5, 5), 0)
med_sp  = cv2.medianBlur(noisy_sp, 5)
med_g   = cv2.medianBlur(noisy_gaussian, 5)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0,0].imshow(low_sp, cmap="gray");  axes[0,0].set_title("Low-Pass (S&P)");      axes[0,0].axis("off")
axes[0,1].imshow(low_g, cmap="gray");   axes[0,1].set_title("Low-Pass (Gaussian)"); axes[0,1].axis("off")
axes[1,0].imshow(med_sp, cmap="gray");  axes[1,0].set_title("Median (S&P)");        axes[1,0].axis("off")
axes[1,1].imshow(med_g, cmap="gray");   axes[1,1].set_title("Median (Gaussian)");   axes[1,1].axis("off")
plt.show()


## 4. High-pass filter

In [ ]:
kernel_hp = np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]])
high_pass = cv2.filter2D(A, -1, kernel_hp)
high_pass_mean = cv2.add(high_pass, A)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(high_pass, cmap="gray"); axes[0].set_title("High-Pass (edges only)"); axes[0].axis("off")
axes[1].imshow(high_pass_mean, cmap="gray"); axes[1].set_title("High-Pass + Original"); axes[1].axis("off")
plt.show()


## 5. Sharpening filter

In [ ]:
kernel_sharp = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
sharpened = cv2.filter2D(A, -1, kernel_sharp)

plt.figure(figsize=(6, 6))
plt.imshow(sharpened, cmap="gray")
plt.title("Sharpened")
plt.axis("off")
plt.show()


## 6. Fourier transform of the cameraman image

In [ ]:
cameraman = cv2.imread("cameraman.jpg", cv2.IMREAD_GRAYSCALE)
f_cam = np.fft.fftshift(np.fft.fft2(cameraman))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(20*np.log(np.abs(f_cam)+1), cmap="gray"); axes[0].set_title("Magnitude"); axes[0].axis("off")
axes[1].imshow(np.angle(f_cam), cmap="gray"); axes[1].set_title("Phase"); axes[1].axis("off")
plt.suptitle("Cameraman - Fourier Transform")
plt.show()


## 7. Fourier transform of the trui image

In [ ]:
trui = cv2.imread("trui.png", cv2.IMREAD_GRAYSCALE)
f_trui = np.fft.fftshift(np.fft.fft2(trui))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(20*np.log(np.abs(f_trui)+1), cmap="gray"); axes[0].set_title("Magnitude"); axes[0].axis("off")
axes[1].imshow(np.angle(f_trui), cmap="gray"); axes[1].set_title("Phase"); axes[1].axis("off")
plt.suptitle("Trui - Fourier Transform")
plt.show()


## 8. Cross-synthesis: cameraman magnitude + trui phase

Phase encodes spatial structure, so the result should resemble the trui image.

In [ ]:
combined = np.abs(f_cam) * np.exp(1j * np.angle(f_trui))
result = np.abs(np.fft.ifft2(np.fft.ifftshift(combined)))

plt.figure(figsize=(6, 6))
plt.imshow(result, cmap="gray")
plt.title("Cameraman magnitude + Trui phase")
plt.axis("off")
plt.show()
